<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep — Audio Stem Separator Demo

Welcome to the **vsep** interactive demo! Separate audio into stems (vocals, drums, bass, instruments) using state-of-the-art AI models.

## Features
- **Fast Processing** — GPU-accelerated inference on Colab
- **Multiple Models** — 21 curated models across 6 categories
- **Leaderboard** — Browse top models ranked by SDR quality
- **Easy to Use** — Upload, pick a model, separate, download

See the [full documentation](https://github.com/BF667-IDLE/vsep/tree/main/docs) for more details.

---

## 1. Setup

Install vsep and its dependencies. This takes about 2–3 minutes.

In [ ]:
#@title Install vsep
#@markdown Run this cell to install vsep (first time only)

# Clone the repository
!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep

# Install dependencies
!pip install -q -r /content/vsep/requirements.txt

# Install ffmpeg (required by pydub for audio I/O)
!apt-get -qq install ffmpeg > /dev/null 2>&1

import os
os.chdir('/content/vsep')

import torch
if torch.cuda.is_available():
    print(f"NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, will use CPU (slower)")
    print("Tip: Go to Runtime > Change runtime type > Hardware accelerator > GPU")

## 2. Upload Your Audio

Upload the audio file you want to separate. Supported formats: MP3, WAV, FLAC, OGG, M4A

In [ ]:
#@title Upload Audio File
#@markdown Upload your audio file for separation

from google.colab import files
import os

print("Please upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f"\nUploaded: {audio_file}")
    os.makedirs("output", exist_ok=True)
else:
    print("No file uploaded. Please run this cell again.")

## 3. Model Leaderboard

Browse the top models ranked by quality. Select a category, then pick a model.

In [ ]:
#@title Model Leaderboard
#@markdown Browse top models by category. Use the dropdown to filter.

MODEL_CATALOG = {
    # --- Vocals (best vocal isolation) ---
    "BS Roformer EP 317 SDR 12.98": {
        "filename": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
        "arch": "Roformer",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 12.98,
    },
    "Mel Roformer Viperx SDR 12.61": {
        "filename": "Mel-Roformer-Viperx-1053.ckpt",
        "arch": "Roformer",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 12.61,
    },
    "BS Roformer Viperx 1297": {
        "filename": "BS-Roformer-Viperx-1297.ckpt",
        "arch": "Roformer",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 12.50,
    },
    "MDX23C InstVoc HQ SDR 11.95": {
        "filename": "MDX23C-8KFFT-InstVoc_HQ.ckpt",
        "arch": "MDXC",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 11.95,
    },
    "MDX Net Inst HQ 2 SDR 10.93": {
        "filename": "UVR-MDX-NET-Inst_HQ_2.onnx",
        "arch": "MDX-Net",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 10.93,
    },
    "MDX Net Inst 1 SDR 10.65": {
        "filename": "UVR-MDX-NET-Inst_1.onnx",
        "arch": "MDX-Net",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 10.65,
    },
    "MDX Net Inst HQ 4 SDR 10.49": {
        "filename": "UVR-MDX-NET-Inst_HQ_4.onnx",
        "arch": "MDX-Net",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 10.49,
    },
    "Kim Vocal 2": {
        "filename": "Kim_Vocal_2.onnx",
        "arch": "MDX-Net",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 10.21,
    },
    "MDX Net Karaoke 2": {
        "filename": "UVR-MDX-NET-KARA_2.onnx",
        "arch": "MDX-Net",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 9.50,
    },
    "VR 5 HP": {
        "filename": "5_HP-UVR.pth",
        "arch": "VR",
        "category": "Vocals",
        "stems": "vocals, instrumental",
        "sdr": 8.22,
    },

    # --- Multi-Stem (4+ stems) ---
    "Demucs v4 Fine Tuned": {
        "filename": "ht-demucs_ft.yaml",
        "arch": "Demucs",
        "category": "Multi-Stem",
        "stems": "vocals, drums, bass, other",
        "sdr": 11.27,
    },
    "Demucs v4 6 Stem": {
        "filename": "htdemucs_6s.yaml",
        "arch": "Demucs",
        "category": "Multi-Stem",
        "stems": "vocals, drums, bass, other, guitar, piano",
        "sdr": 10.80,
    },

    # --- Reverb / Echo Removal ---
    "Reverb HQ SDR 11.20": {
        "filename": "MDX23C-8KFFT-Reverb_HQ.ckpt",
        "arch": "MDXC",
        "category": "Reverb/Echo Removal",
        "stems": "vocals, no reverb",
        "sdr": 11.20,
    },
    "Echo HQ SDR 11.10": {
        "filename": "MDX23C-8KFFT-Echo_HQ.ckpt",
        "arch": "MDXC",
        "category": "Reverb/Echo Removal",
        "stems": "vocals, no echo",
        "sdr": 11.10,
    },

    # --- De-Esser ---
    "De-Esser MDX SDR 10.90": {
        "filename": "MDX23C-8KFFT-DeEsser.ckpt",
        "arch": "MDXC",
        "category": "De-Esser",
        "stems": "vocals, no sibilance",
        "sdr": 10.90,
    },

    # --- Stem Isolation ---
    "Guitar Isolation SDR 10.80": {
        "filename": "MDX23C-8KFFT-Guitar.ckpt",
        "arch": "MDXC",
        "category": "Stem Isolation",
        "stems": "guitar, no guitar",
        "sdr": 10.80,
    },
    "Drum Isolation SDR 10.50": {
        "filename": "MDX23C-8KFFT-Drum.ckpt",
        "arch": "MDXC",
        "category": "Stem Isolation",
        "stems": "drums, no drums",
        "sdr": 10.50,
    },
    "Bass Isolation SDR 10.10": {
        "filename": "MDX23C-8KFFT-Bass.ckpt",
        "arch": "MDXC",
        "category": "Stem Isolation",
        "stems": "bass, no bass",
        "sdr": 10.10,
    },
    "Piano Isolation SDR 9.90": {
        "filename": "MDX23C-8KFFT-Piano.ckpt",
        "arch": "MDXC",
        "category": "Stem Isolation",
        "stems": "piano, no piano",
        "sdr": 9.90,
    },

    # --- Instrumental / Karaoke ---
    "MDX23C InstVoc SDR 10.75": {
        "filename": "MDX23C-8KFFT-InstVoc.ckpt",
        "arch": "MDXC",
        "category": "Instrumental",
        "stems": "vocals, instrumental",
        "sdr": 10.75,
    },
}

# Build categories
categories = {}
for name, info in MODEL_CATALOG.items():
    cat = info["category"]
    if cat not in categories:
        categories[cat] = []
    categories[cat].append((name, info))

# Sort each category by SDR
for cat in categories:
    categories[cat].sort(key=lambda x: x[1]["sdr"], reverse=True)

category = "Vocals" #@param ["Vocals", "Multi-Stem", "Reverb/Echo Removal", "De-Esser", "Stem Isolation", "Instrumental"]

models_in_cat = categories[category]

print(f"\n{'#':<4} {'Model':<40} {'Arch':<10} {'SDR':>6}  {'Stems'}")
print("=" * 90)
for i, (name, info) in enumerate(models_in_cat, 1):
    print(f"{i:<4} {name:<40} {info['arch']:<10} {info['sdr']:>6.2f}  {info['stems']}")

print(f"\n{len(models_in_cat)} models in '{category}' | Total catalog: {len(MODEL_CATALOG)} models")



## 4. Choose Model

Select the model you want to use. Pick from the dropdown or type a custom filename.

In [ ]:
#@title Select Model
#@markdown Pick from the leaderboard or enter a custom filename

sorted_names = sorted(MODEL_CATALOG.keys(), key=lambda k: MODEL_CATALOG[k]["sdr"], reverse=True)

model_display_name = "BS Roformer EP 317 SDR 12.98" #@param ["BS Roformer EP 317 SDR 12.98", "Mel Roformer Viperx SDR 12.61", "BS Roformer Viperx 1297", "Demucs v4 Fine Tuned", "MDX23C InstVoc HQ SDR 11.95", "Reverb HQ SDR 11.20", "Echo HQ SDR 11.10", "De-Esser MDX SDR 10.90", "MDX Net Inst HQ 2 SDR 10.93", "MDX Net Inst 1 SDR 10.65", "Guitar Isolation SDR 10.80", "Drum Isolation SDR 10.50", "Bass Isolation SDR 10.10", "MDX23C InstVoc SDR 10.75", "Piano Isolation SDR 9.90", "MDX Net Inst HQ 4 SDR 10.49", "Kim Vocal 2", "MDX Net Karaoke 2", "Demucs v4 6 Stem", "VR 5 HP", "(custom filename)"]

if model_display_name == "(custom filename)":
    custom_name = "model_bs_roformer_ep_317_sdr_12.9755.ckpt" #@param {type:"string"}
    selected_model = custom_name
    print(f"Using custom model: {selected_model}")
else:
    selected_model = MODEL_CATALOG[model_display_name]["filename"]
    info = MODEL_CATALOG[model_display_name]
    print(f"Model:    {model_display_name}")
    print(f"Filename: {selected_model}")
    print(f"Arch:     {info['arch']}")
    print(f"Category: {info['category']}")
    print(f"Stems:    {info['stems']}")
    print(f"SDR:      {info['sdr']}")



## 5. Separate Audio

Run the separation on your uploaded audio file.

In [ ]:
#@title Run Separation
#@markdown Click to start the separation process

from separator import Separator
import config.variables as cfg

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("Starting separation process...")
print(f"Input file: {audio_file}")
print(f"Model: {selected_model}")
print("\nThis may take 2-5 minutes depending on the model and audio length...\n")

# Initialize separator
separator = Separator(
    model_file_dir="/content/vsep/models",
    output_dir="/content/vsep/output",
    output_format="WAV",
    use_soundfile=True,
)

# Load model
separator.load_model(model_filename=selected_model)

# Run separation
output_files = separator.separate(audio_file)

print(f"\nSeparation complete!")
print(f"Output files: {len(output_files)}")
for f in output_files:
    print(f"  - {f}")



## 6. Listen to Results

Play back the separated stems to hear the results.

In [ ]:
#@title Listen to Separated Stems
#@markdown Browse and play the separated audio files

import glob
from IPython.display import Audio, display

output_files = sorted(glob.glob("/content/vsep/output/*.*"))

if output_files:
    print(f"Found {len(output_files)} separated stems:\n")
    for i, file_path in enumerate(output_files, 1):
        filename = os.path.basename(file_path)
        print(f"{i}. {filename}")
        display(Audio(file_path))
        print("-" * 50)
else:
    print("No output files found. Please run the separation first.")



## 7. Download Results

Download the separated stems to your computer.

In [ ]:
#@title Download Separated Stems
#@markdown Download all separated audio files

from google.colab import files
import glob

output_files = glob.glob("/content/vsep/output/*.*")

if output_files:
    print(f"Downloading {len(output_files)} files...")
    for file_path in output_files:
        files.download(file_path)
        print(f"Downloaded: {os.path.basename(file_path)}")
else:
    print("No files to download. Please run separation first.")



In [ ]:
#@title Browse Full Model List (Optional)
#@markdown Run this to see all 100+ supported models from UVR

from separator import Separator

sep = Separator(info_only=True)
models = sep.get_simplified_model_list(filter_sort_by="vocals")

print(f"Showing top 20 vocal models (out of {len(models)} total)\n")
print(f"{'#':<4} {'Model':<55} {'Arch':<10} {'Stems'}")
print("-" * 100)
for i, (filename, info) in enumerate(list(models.items())[:20], 1):
    stems = ", ".join(info["Stems"])
    print(f"{i:<4} {filename:<55} {info['Type']:<10} {stems}")

## Additional Information

### Quick Recommendations

| What You Need | Best Model | Why |
|:---------------|:-----------|:-----|
| Clean vocals | BS Roformer EP 317 | Highest vocal SDR (12.98) |
| All 4 stems | Demucs v4 Fine Tuned | Vocals + drums + bass + other |
| 6 stems | Demucs v4 6 Stem | Adds guitar and piano |
| Karaoke track | MDX Net Inst HQ 2 | Clean instrumental |
| Remove reverb | Reverb HQ | Isolates dry vocals |
| Remove echo | Echo HQ | Removes room echo |
| Remove sibilance | De-Esser MDX | De-esses vocals |
| Isolate guitar | Guitar Isolation | Extracts guitar |
| Isolate drums | Drum Isolation | Extracts drums |
| Isolate bass | Bass Isolation | Extracts bass |
| Isolate piano | Piano Isolation | Extracts piano |

### Tips

1. **GPU Acceleration:** Make sure you're using a GPU runtime for faster processing
   - Go to `Runtime > Change runtime type > Hardware accelerator > GPU`
2. **Long Files:** For files longer than 10 minutes, enable chunking to avoid memory errors
3. **Quality vs Speed:** Larger Roformer models produce better quality but take longer
4. **Memory:** If you run out of memory, try a smaller model (MDX-Net) or shorter audio

### Troubleshooting

**"No GPU detected"**
- Go to Runtime > Change runtime type > GPU
- Restart the runtime if needed

**"Download failed"**
- The model download may have timed out — re-run the separation cell (downloads are cached)
- Increase timeout: `cfg.DOWNLOAD_TIMEOUT = 600`

**"Out of memory"**
- Try a shorter audio file or use `chunk_duration=300` when creating the Separator
- Use a simpler model (MDX-Net instead of Roformer)

**"Module not found"**
- Make sure you ran the installation cell first
- Try restarting the runtime (Runtime > Restart runtime)

---

### Links

- **GitHub:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- **API Reference:** [docs/API-Reference.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/API-Reference.md)
- **Architecture:** [docs/Architecture.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/Architecture.md)
- **Install Guide:** [INSTALL.md](https://github.com/BF667-IDLE/vsep/blob/main/INSTALL.md)
- **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)

Made with care using vsep